# Client Math Student Evaluation with vLLM

Evaluate the trained Math Student on AIME25 and AIME26 with the OPSD main evaluation protocol. One vLLM engine batches requests on a single GPU; the two benchmarks run sequentially through the same loaded model. Results are written incrementally to JSONL and can be resumed.

Protocol: Thinking ON, Avg@12, temperature 1.0, no top-p truncation, top-k disabled, and up to 38912 generated tokens.


In [ ]:
# Papermill parameters
GPU_ID = "0"
MODEL_DIR = "results/client_math_opsd_qwen3_4b_fullft/standalone_math_fullft_100steps_single_4090"
OUTPUT_DIR = "results/client_math_opsd_qwen3_4b_fullft/client_math_avg12_evaluation_vllm"
LOCAL_FILES_ONLY = True
SEED = 42

SAMPLES_PER_PROMPT = 12
MAX_MODEL_LEN = 40960
MAX_NEW_TOKENS = 38912
TEMPERATURE = 1.0
TOP_P = 1.0
TOP_K = -1
MIN_P = 0.0
PRESENCE_PENALTY = 0.0
ENABLE_THINKING = True

VLLM_GPU_MEMORY_UTILIZATION = 0.85
VLLM_MAX_NUM_SEQS = 12
VLLM_REQUEST_BATCH_SIZE = 4
VLLM_ENFORCE_EAGER = True

RUN_AIME25 = True
RUN_AIME26 = True
LIMIT_AIME25 = None
LIMIT_AIME26 = None
RESUME = True


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

import hashlib
import importlib.metadata
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from datasets import DownloadConfig, disable_progress_bars, load_dataset
from math_verify import parse as math_parse, verify as math_verify
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

workspace = Path.cwd().resolve()
model_dir = (workspace / MODEL_DIR).resolve()
output_dir = (workspace / OUTPUT_DIR).resolve()
raw_dir = output_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

if not (model_dir / "config.json").is_file():
    raise FileNotFoundError(f"Model directory is incomplete: {model_dir}")
if int(SAMPLES_PER_PROMPT) != 12:
    raise ValueError("This notebook is configured for OPSD Avg@12 evaluation.")
if int(MAX_MODEL_LEN) <= int(MAX_NEW_TOKENS):
    raise ValueError("MAX_MODEL_LEN must leave room for the input prompt.")

packages = {}
for package in ["vllm", "torch", "transformers", "datasets", "math-verify"]:
    try:
        packages[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        packages[package] = None

print("Packages:", json.dumps(packages, ensure_ascii=False), flush=True)
print(f"Model:  {model_dir}", flush=True)
print(f"Output: {output_dir}", flush=True)


In [ ]:
disable_progress_bars()
download_config = DownloadConfig(local_files_only=bool(LOCAL_FILES_ONLY))
datasets_to_run = {}
if RUN_AIME25:
    dataset = load_dataset("math-ai/aime25", split="test", download_config=download_config)
    if LIMIT_AIME25 is not None:
        dataset = dataset.select(range(min(int(LIMIT_AIME25), len(dataset))))
    datasets_to_run["aime25"] = dataset
if RUN_AIME26:
    dataset = load_dataset("math-ai/aime26", split="test", download_config=download_config)
    if LIMIT_AIME26 is not None:
        dataset = dataset.select(range(min(int(LIMIT_AIME26), len(dataset))))
    datasets_to_run["aime26"] = dataset
if not datasets_to_run:
    raise ValueError("At least one benchmark must be enabled.")
for benchmark, dataset in datasets_to_run.items():
    if not {"problem", "answer"}.issubset(dataset.column_names):
        raise ValueError(f"Unexpected columns for {benchmark}: {dataset.column_names}")
    print(f"{benchmark}: {len(dataset)} problems", flush=True)

tokenizer = AutoTokenizer.from_pretrained(
    model_dir,
    local_files_only=bool(LOCAL_FILES_ONLY),
    trust_remote_code=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading one vLLM engine...", flush=True)
llm = LLM(
    model=str(model_dir),
    tokenizer=str(model_dir),
    dtype="bfloat16",
    tensor_parallel_size=1,
    max_model_len=int(MAX_MODEL_LEN),
    gpu_memory_utilization=float(VLLM_GPU_MEMORY_UTILIZATION),
    max_num_seqs=int(VLLM_MAX_NUM_SEQS),
    trust_remote_code=True,
    enforce_eager=bool(VLLM_ENFORCE_EAGER),
    enable_prefix_caching=True,
    seed=int(SEED),
)
print("vLLM engine ready.", flush=True)


In [ ]:
def format_seconds(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def append_jsonl(path, record):
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def record_key(sample_id, sample_index):
    return f"{sample_id}::sample-{int(sample_index):02d}"


def extract_last_boxed(text):
    starts = [match.start() for match in re.finditer(r"\\boxed\s*\{", str(text))]
    for start in reversed(starts):
        brace_start = text.find("{", start)
        depth = 0
        for index in range(brace_start, len(text)):
            if text[index] == "{":
                depth += 1
            elif text[index] == "}":
                depth -= 1
                if depth == 0:
                    return text[brace_start + 1:index].strip()
    return None


def normalize_answer(value):
    value = str(value).strip().replace("$", "").replace(",", "")
    value = re.sub(r"\\(?:mathrm|text)\{([^{}]*)\}", r"\1", value)
    value = re.sub(r"\s+", "", value)
    if re.fullmatch(r"[+-]?\d+\.0+", value):
        value = str(int(float(value)))
    if re.fullmatch(r"[+-]?\d+", value):
        value = str(int(value))
    return value


def grade_boxed_answer(predicted, gold):
    if predicted is None:
        return False
    try:
        predicted_text = predicted if "$" in predicted else f"${predicted}$"
        gold_text = str(gold) if "$" in str(gold) else f"${gold}$"
        parsed_prediction = math_parse(predicted_text, fallback_mode="no_fallback")
        parsed_gold = math_parse(gold_text, fallback_mode="no_fallback")
        return bool(math_verify(parsed_gold, parsed_prediction, timeout_seconds=5))
    except Exception:
        return normalize_answer(predicted) == normalize_answer(gold)


def split_thinking(text):
    text = str(text).strip()
    if "</think>" in text:
        thinking, final_answer = text.rsplit("</think>", 1)
        thinking = re.sub(r"^\s*<think>\s*", "", thinking).strip()
        return thinking, final_answer.strip(), True
    return "", text, False


def stable_request_seed(benchmark, sample_id):
    digest = hashlib.sha256(f"{benchmark}:{sample_id}".encode("utf-8")).digest()
    return (int(SEED) + int.from_bytes(digest[:4], "big")) % (2**31 - 1)


def model_signature(path):
    digest = hashlib.sha256()
    for candidate in sorted(Path(path).glob("*.safetensors")):
        stat = candidate.stat()
        digest.update(f"{candidate.name}:{stat.st_size}:{stat.st_mtime_ns}".encode("utf-8"))
    for name in ["config.json", "model.safetensors.index.json"]:
        candidate = Path(path) / name
        if candidate.is_file():
            digest.update(candidate.read_bytes())
    return digest.hexdigest()[:16]


In [ ]:
protocol = {
    "backend": "vllm",
    "metric": "avg_at_12",
    "samples_per_prompt": int(SAMPLES_PER_PROMPT),
    "max_model_len": int(MAX_MODEL_LEN),
    "max_new_tokens": int(MAX_NEW_TOKENS),
    "temperature": float(TEMPERATURE),
    "top_p": float(TOP_P),
    "top_k": int(TOP_K),
    "min_p": float(MIN_P),
    "presence_penalty": float(PRESENCE_PENALTY),
    "enable_thinking": bool(ENABLE_THINKING),
    "seed": int(SEED),
    "model_signature": model_signature(model_dir),
    "vllm_version": packages["vllm"],
}
RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(protocol, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
print(f"Run fingerprint: {RUN_FINGERPRINT}", flush=True)
print(json.dumps(protocol, ensure_ascii=False, indent=2), flush=True)


def load_matching_records(path):
    records = {}
    path = Path(path)
    if not path.is_file():
        return records
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                print(f"[WARN] Ignoring truncated line {path}:{line_number}", flush=True)
                continue
            if record.get("run_fingerprint") == RUN_FINGERPRINT:
                key = record_key(record["sample_id"], record["sample_index"])
                records[key] = record
    return records


def render_prompt(problem):
    user_message = (
        f"{str(problem).strip()}\n\n"
        "Please reason step by step, and put your final answer within \\boxed{}."
    )
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_message}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=bool(ENABLE_THINKING),
    )


def make_sampling_params(max_tokens, seed):
    return SamplingParams(
        n=int(SAMPLES_PER_PROMPT),
        temperature=float(TEMPERATURE),
        top_p=float(TOP_P),
        top_k=int(TOP_K),
        min_p=float(MIN_P),
        presence_penalty=float(PRESENCE_PENALTY),
        max_tokens=int(max_tokens),
        seed=int(seed),
        skip_special_tokens=True,
    )


In [ ]:
def run_benchmark(benchmark, dataset):
    output_path = raw_dir / f"{benchmark}.jsonl"
    records = load_matching_records(output_path) if RESUME else {}
    completed = {key for key, record in records.items() if record.get("status") == "ok"}
    entries = []
    expected = set()
    for index, row in enumerate(dataset):
        sample_id = str(row.get("id", index))
        keys = {record_key(sample_id, sample_index) for sample_index in range(SAMPLES_PER_PROMPT)}
        expected.update(keys)
        if not keys.issubset(completed):
            entries.append({
                "sample_id": sample_id,
                "problem": str(row["problem"]),
                "gold_answer": str(row["answer"]),
            })

    total = len(expected)
    done = len(expected.intersection(completed))
    if not entries:
        print(f"[SKIP] {benchmark}/avg@12: {done}/{total} generations complete", flush=True)
        return records
    if done:
        print(f"[RESUME] {benchmark}/avg@12: {done}/{total} generations complete", flush=True)

    benchmark_started = time.time()
    for chunk_start in range(0, len(entries), int(VLLM_REQUEST_BATCH_SIZE)):
        chunk = entries[chunk_start:chunk_start + int(VLLM_REQUEST_BATCH_SIZE)]
        prompts = []
        sampling_params = []
        prepared = []
        for entry in chunk:
            rendered = render_prompt(entry["problem"])
            prompt_ids = tokenizer(rendered, add_special_tokens=False)["input_ids"]
            available = int(MAX_MODEL_LEN) - len(prompt_ids)
            if available < 1:
                raise ValueError(
                    f"{benchmark}:{entry['sample_id']} prompt has {len(prompt_ids)} tokens, "
                    f"exceeding MAX_MODEL_LEN={MAX_MODEL_LEN}"
                )
            effective_max_tokens = min(int(MAX_NEW_TOKENS), available)
            request_seed = stable_request_seed(benchmark, entry["sample_id"])
            prompts.append(rendered)
            sampling_params.append(make_sampling_params(effective_max_tokens, request_seed))
            prepared.append({
                **entry,
                "input_tokens": len(prompt_ids),
                "effective_max_new_tokens": effective_max_tokens,
                "request_seed": request_seed,
            })

        chunk_started = time.time()
        outputs = llm.generate(prompts, sampling_params=sampling_params, use_tqdm=True)
        chunk_seconds = time.time() - chunk_started
        if len(outputs) != len(prepared):
            raise RuntimeError(f"vLLM returned {len(outputs)} requests for {len(prepared)} prompts")

        for entry, request_output in zip(prepared, outputs):
            choices = sorted(request_output.outputs, key=lambda choice: int(choice.index))
            if len(choices) != int(SAMPLES_PER_PROMPT):
                raise RuntimeError(
                    f"{benchmark}:{entry['sample_id']} returned {len(choices)} outputs, "
                    f"expected {SAMPLES_PER_PROMPT}"
                )
            for choice in choices:
                sample_index = int(choice.index)
                key = record_key(entry["sample_id"], sample_index)
                if key in completed:
                    continue
                raw_generation = str(choice.text)
                thinking, final_answer, found_think_close = split_thinking(raw_generation)
                predicted_answer = extract_last_boxed(raw_generation)
                record = {
                    "run_fingerprint": RUN_FINGERPRINT,
                    "backend": "vllm",
                    "benchmark": benchmark,
                    "sample_id": entry["sample_id"],
                    "sample_index": sample_index,
                    "samples_per_prompt": int(SAMPLES_PER_PROMPT),
                    "problem": entry["problem"],
                    "gold_answer": entry["gold_answer"],
                    "request_seed": entry["request_seed"],
                    "raw_generation": raw_generation,
                    "thinking": thinking,
                    "final_answer": final_answer,
                    "found_think_close": found_think_close,
                    "predicted_answer": predicted_answer or "",
                    "formatted": predicted_answer is not None,
                    "correct": grade_boxed_answer(predicted_answer, entry["gold_answer"]),
                    "input_tokens": entry["input_tokens"],
                    "output_tokens": len(choice.token_ids),
                    "effective_max_new_tokens": entry["effective_max_new_tokens"],
                    "finish_reason": choice.finish_reason,
                    "chunk_generation_seconds": chunk_seconds,
                    "status": "ok",
                }
                append_jsonl(output_path, record)
                records[key] = record
                completed.add(key)
                done += 1
            percentage = 100.0 * done / total
            print(
                f"[PROGRESS] {benchmark}/avg@12: {done}/{total} ({percentage:.2f}%) "
                f"elapsed={format_seconds(time.time() - benchmark_started)}",
                flush=True,
            )

    final_records = load_matching_records(output_path)
    successful = {key for key, record in final_records.items() if record.get("status") == "ok"}
    missing = expected - successful
    if missing:
        raise RuntimeError(f"{benchmark} evaluation incomplete: {len(missing)} generations missing")
    return final_records


benchmark_records = {}
for benchmark, dataset in datasets_to_run.items():
    benchmark_records[benchmark] = run_benchmark(benchmark, dataset)


In [ ]:
metric_rows = []
per_problem_rows = []
for benchmark, dataset in datasets_to_run.items():
    records = benchmark_records[benchmark]
    correct_generations = 0
    formatted_generations = 0
    passed_problems = 0
    for index, row in enumerate(dataset):
        sample_id = str(row.get("id", index))
        problem_records = [records[record_key(sample_id, j)] for j in range(SAMPLES_PER_PROMPT)]
        problem_correct = sum(bool(record.get("correct")) for record in problem_records)
        problem_formatted = sum(bool(record.get("formatted")) for record in problem_records)
        correct_generations += problem_correct
        formatted_generations += problem_formatted
        passed_problems += int(problem_correct > 0)
        per_problem_rows.append({
            "benchmark": benchmark.upper(),
            "sample_id": sample_id,
            "correct_samples": problem_correct,
            "formatted_samples": problem_formatted,
            "total_samples": int(SAMPLES_PER_PROMPT),
            "average_accuracy": problem_correct / int(SAMPLES_PER_PROMPT),
        })
    total_generations = len(dataset) * int(SAMPLES_PER_PROMPT)
    metric_rows.append({
        "benchmark": benchmark.upper(),
        "metric": "avg_at_12",
        "problems": len(dataset),
        "samples_per_prompt": int(SAMPLES_PER_PROMPT),
        "correct_generations": correct_generations,
        "total_generations": total_generations,
        "score": correct_generations / total_generations,
        "score_percent": 100.0 * correct_generations / total_generations,
        "pass_at_12_problems": passed_problems,
        "pass_at_12_percent": 100.0 * passed_problems / len(dataset),
        "formatted_generations": formatted_generations,
        "format_rate_percent": 100.0 * formatted_generations / total_generations,
    })

metrics = pd.DataFrame(metric_rows)
per_problem = pd.DataFrame(per_problem_rows)
metrics_csv = output_dir / "avg_at_12_metrics.csv"
metrics_json = output_dir / "avg_at_12_metrics.json"
per_problem_csv = output_dir / "avg_at_12_per_problem.csv"
per_problem_json = output_dir / "avg_at_12_per_problem.json"
chart_path = output_dir / "aime25_aime26_avg_at_12_vllm.png"
metrics.to_csv(metrics_csv, index=False, encoding="utf-8-sig")
per_problem.to_csv(per_problem_csv, index=False, encoding="utf-8-sig")
write_json(per_problem_json, per_problem_rows)
write_json(metrics_json, {
    "model": str(model_dir),
    "run_fingerprint": RUN_FINGERPRINT,
    "protocol": protocol,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "metrics": metric_rows,
})

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(7.5, 5.5))
bars = axis.bar(
    metrics["benchmark"],
    metrics["score_percent"],
    color=["#4C78A8", "#72B7B2"][:len(metrics)],
    width=0.58,
    edgecolor="#333333",
)
for bar, row in zip(bars, metric_rows):
    axis.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{row['score_percent']:.2f}%\n({row['correct_generations']}/{row['total_generations']})",
        ha="center",
        va="bottom",
    )
axis.set_ylabel("Avg@12 accuracy (%)")
axis.set_title("Full-parameter OPSD Math Student — vLLM Avg@12")
axis.grid(axis="y", linestyle="--", alpha=0.3)
axis.set_ylim(0, max(10.0, float(metrics['score_percent'].max()) * 1.25 + 1.0))
figure.tight_layout()
figure.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()

display(metrics)
print(f"Metrics:     {metrics_json}", flush=True)
print(f"Per problem: {per_problem_json}", flush=True)
print(f"Chart:       {chart_path}", flush=True)


## Papermill execution

Run this notebook with the dedicated vLLM environment. The main Papermill log contains progress for both benchmarks because only one vLLM engine is used.
